In [1]:
import pandas as pd
from pathlib import Path
DATA_DIR = Path.cwd() / "data" / "RHEOLOGY"
if not DATA_DIR.is_dir():
    DATA_DIR = Path.cwd() / "Rheology" / "data" / "RHEOLOGY"
train_df = pd.read_json(
    DATA_DIR / "train.json",
    lines=True,
)

dev_df = pd.read_json(
    DATA_DIR / "dev.json",
    lines=True,
)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)

display(train_df.head())

Train: (1612, 3)
Dev: (403, 3)


,id,sentence,label
0,0,"Kim et al., 2006b), has conduced to confused i...",BIODEGRAD_PROP
1,1,A similar type of physical damage in the hydro...,BIODEGRAD_POLY
2,2,"At pH 1 and 9, the overall observations were s...",BIODEGRAD_PROP
3,3,"As expected, the kinetic constants showed a pr...",BIODEGRAD_PROP
4,4,This can be observed from Figure 9. The cracks...,BIODEGRAD_PROP


In [2]:
print(train_df["label"].value_counts())
print(dev_df["label"].value_counts())

label
BIODEGRAD_POLY      553
POLY_STRUC          470
BIODEGRAD_PROP      421
MECHANICAL_PROP      90
RHEOLOGICAL_PROP     78
Name: count, dtype: int64
label
BIODEGRAD_POLY      151
BIODEGRAD_PROP      105
POLY_STRUC           97
MECHANICAL_PROP      31
RHEOLOGICAL_PROP     19
Name: count, dtype: int64


In [3]:
# 仅检测，不自动删除或填充数据。
def inspect_split(frame, split_name):
    print(f"\n=== {split_name} ===")
    print("Shape:", frame.shape)
    print("NaN counts:")
    display(frame.isna().sum().to_frame("NaN"))
    print("Empty/whitespace sentence:", frame["sentence"].fillna("").astype(str).str.strip().eq("").sum())
    print("Empty/whitespace label:", frame["label"].fillna("").astype(str).str.strip().eq("").sum())
    print("Exact duplicate rows (after first):", frame[["id", "sentence", "label"]].duplicated().sum())
    print("Repeated IDs (after first):", frame["id"].duplicated().sum())
    # 忽略 id，检查相同文本和标签；规范化空白仅用于检测。
    text_key = frame["sentence"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    keys = pd.DataFrame({"text": text_key, "label": frame["label"]})
    valid = text_key.notna() & text_key.ne("")
    print("Repeated normalized sentences (after first):", text_key.loc[valid].duplicated().sum())
    print("Repeated sentence-label pairs (after first):", keys.loc[valid].duplicated().sum())
    conflicts = keys.loc[valid].groupby("text")["label"].nunique()
    print("Sentences with conflicting labels:", conflicts.gt(1).sum())
    repeated = valid & text_key.duplicated(keep=False)
    if repeated.any():
        display(frame.loc[repeated, ["id", "sentence", "label"]].head(10))
    return text_key

train_text_keys = inspect_split(train_df, "train")
dev_text_keys = inspect_split(dev_df, "dev")
shared_texts = set(train_text_keys.dropna()) & set(dev_text_keys.dropna())
shared_texts.discard("")
print("\nDistinct normalized sentences shared by train/dev:", len(shared_texts))
if shared_texts:
    display(train_df.loc[train_text_keys.isin(shared_texts), ["id", "sentence", "label"]].head(10))



=== train ===
Shape: (1612, 3)
NaN counts:


,NaN
id,0
sentence,0
label,0


Empty/whitespace sentence: 0
Empty/whitespace label: 0
Exact duplicate rows (after first): 0
Repeated IDs (after first): 0
Repeated normalized sentences (after first): 6
Repeated sentence-label pairs (after first): 6
Sentences with conflicting labels: 0


,id,sentence,label
230,230,Analysis of the molecular weight profile durin...,BIODEGRAD_PROP
248,248,"1H‐NMR, DSC, FTIR, XRD, and SEM. A faster rate...",BIODEGRAD_PROP
313,313,Analysis of the molecular weight profile durin...,BIODEGRAD_PROP
619,619,Most biodegradable polymers belong to thermopl...,BIODEGRAD_POLY
643,643,5 resulted in the formation of dimers. From th...,BIODEGRAD_PROP
745,745,"1H‐NMR, DSC, FTIR, XRD, and SEM. A faster rate...",BIODEGRAD_PROP
1092,1092,There was also an insignificant mass loss of t...,BIODEGRAD_PROP
1163,1163,Most biodegradable polymers belong to thermopl...,BIODEGRAD_POLY
1171,1171,"Tsuji et al., 2006). Crystallinity is determin...",POLY_STRUC
1275,1275,"Tsuji et al., 2006). Crystallinity is determin...",POLY_STRUC



=== dev ===
Shape: (403, 3)
NaN counts:


,NaN
id,0
sentence,0
label,0


Empty/whitespace sentence: 0
Empty/whitespace label: 0
Exact duplicate rows (after first): 0
Repeated IDs (after first): 0
Repeated normalized sentences (after first): 1
Repeated sentence-label pairs (after first): 1
Sentences with conflicting labels: 0


,id,sentence,label
259,259,and poly (ethylene glycol) (PEG) was studied a...,BIODEGRAD_POLY
318,318,and poly (ethylene glycol) (PEG) was studied a...,BIODEGRAD_POLY



Distinct normalized sentences shared by train/dev: 2


,id,sentence,label
925,925,Thermoplastics from polyolefins are not biodeg...,BIODEGRAD_PROP
1465,1465,"Chandra and Rustgi, 1998, L ö rcks, 1998, Lun...",BIODEGRAD_PROP


In [4]:
# 固定类别顺序：仅从训练集建立映射，验证集沿用。
import numpy as np

if train_df["label"].isna().any() or dev_df["label"].isna().any():
    raise ValueError("Missing labels detected; resolve them before encoding")
if any(frame["label"].astype(str).str.strip().eq("").any() for frame in [train_df, dev_df]):
    raise ValueError("Empty labels detected; resolve them before encoding")
label_names = sorted(train_df["label"].unique().tolist())
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = dict(enumerate(label_names))
NUM_LABELS = len(label_names)
unknown_labels = set(dev_df["label"]) - set(label_names)
if unknown_labels:
    raise ValueError(f"Dev labels absent from train: {sorted(unknown_labels)}")

for frame in [train_df, dev_df]:
    # 单标签多分类：CrossEntropyLoss 使用整数 label_id。
    frame["label_id"] = frame["label"].map(label_to_id).astype(int)
    # 便于查看的向量编码；单标签时每行只有一个 1，即 one-hot。
    frame["labels"] = np.eye(NUM_LABELS, dtype=np.int8)[frame["label_id"].to_numpy()].tolist()

print("Number of classes:", NUM_LABELS)
display(pd.DataFrame({"label": label_names, "label_id": range(NUM_LABELS)}))
display(train_df[["sentence", "label", "label_id", "labels"]].head(10))


Number of classes: 5


,label,label_id
0,BIODEGRAD_POLY,0
1,BIODEGRAD_PROP,1
2,MECHANICAL_PROP,2
3,POLY_STRUC,3
4,RHEOLOGICAL_PROP,4


,sentence,label,label_id,labels
0,"Kim et al., 2006b), has conduced to confused i...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
1,A similar type of physical damage in the hydro...,BIODEGRAD_POLY,0,"[1, 0, 0, 0, 0]"
2,"At pH 1 and 9, the overall observations were s...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
3,"As expected, the kinetic constants showed a pr...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
4,This can be observed from Figure 9. The cracks...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
5,The synthesis of lactic acid into high−molecul...,POLY_STRUC,3,"[0, 0, 0, 1, 0]"
6,Figure 1 Open in figure viewer PowerPoint Resi...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
7,Eq. 5 corresponds to the initial viscosity inc...,RHEOLOGICAL_PROP,4,"[0, 0, 0, 0, 1]"
8,Mass spectroscopy was used to determine the na...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
9,the degradation process can occur during the m...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"


In [5]:
df = train_df.copy()
df.head(10)

,id,sentence,label,label_id,labels
0,0,"Kim et al., 2006b), has conduced to confused i...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
1,1,A similar type of physical damage in the hydro...,BIODEGRAD_POLY,0,"[1, 0, 0, 0, 0]"
2,2,"At pH 1 and 9, the overall observations were s...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
3,3,"As expected, the kinetic constants showed a pr...",BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
4,4,This can be observed from Figure 9. The cracks...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
5,5,The synthesis of lactic acid into high−molecul...,POLY_STRUC,3,"[0, 0, 0, 1, 0]"
6,6,Figure 1 Open in figure viewer PowerPoint Resi...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
7,7,Eq. 5 corresponds to the initial viscosity inc...,RHEOLOGICAL_PROP,4,"[0, 0, 0, 0, 1]"
8,8,Mass spectroscopy was used to determine the na...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"
9,9,the degradation process can occur during the m...,BIODEGRAD_PROP,1,"[0, 1, 0, 0, 0]"


In [6]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from datasets import Dataset, disable_progress_bars
from pathlib import Path
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.utils import logging as transformers_logging

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)


MODELS = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased",
]

# 明确选择 RTX 5080，无法访问时直接停止。
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable: select the project's CUDA-enabled .venv kernel")
gpu_indices = [i for i in range(torch.cuda.device_count()) if "5080" in torch.cuda.get_device_name(i)]
if not gpu_indices:
    raise RuntimeError("RTX 5080 not found among visible CUDA devices")
torch.cuda.set_device(gpu_indices[0])
DEVICE = torch.device("cuda", gpu_indices[0])
print("Training GPU:", torch.cuda.get_device_name(DEVICE), "| Device:", DEVICE)

# 合并原始 train/dev 进行五折；保留来源前缀，避免 split 内 ID 重号。
from sklearn.model_selection import StratifiedKFold
parts = []
for split_name, frame in [("train", train_df), ("dev", dev_df)]:
    part = frame[["id", "sentence", "label_id"]].copy().rename(
        columns={"sentence": "training_text", "label_id": "labels"}
    )
    part["document_id"] = split_name + "_" + part["id"].astype(str)
    part["source_split"] = split_name
    parts.append(part)
training_df = pd.concat(parts, ignore_index=True)
training_df["labels"] = training_df["labels"].astype("int64")
if training_df["training_text"].isna().any() or training_df["training_text"].str.strip().eq("").any():
    raise ValueError("Missing/empty training text")
assert training_df["document_id"].is_unique
training_labels = training_df["labels"].to_numpy(dtype=np.int64)
NUM_LABELS = len(label_names)
PROBLEM_TYPE = "single_label_classification"
assert np.isin(training_labels, range(NUM_LABELS)).all()
N_SPLITS = 5
disable_progress_bars()
transformers_logging.disable_progress_bar()
TRAINING_SEED = 42

NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# 必须与之前EnvDataset一致
MAX_LENGTH = 128

PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "Rheology" / "data" / "RHEOLOGY").is_dir():
    PROJECT_DIR = PROJECT_DIR / "Rheology"
OUTPUT_ROOT = str(PROJECT_DIR / "results" / "rheology_5fold")
RESULT_CSV = os.path.join(OUTPUT_ROOT, "rheology_5fold_results.csv")

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True,
)
# 保存输出位置与类别名称的对应关系，不改变数据。
import json
Path(OUTPUT_ROOT, "label_names.json").write_text(
    json.dumps(label_names, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 只生成一次划分，七个模型共享同一份 train/eval 索引。
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=TRAINING_SEED)
fold_indices = list(splitter.split(training_df["training_text"], training_labels))
fold_assignments = training_df[["document_id", "source_split", "labels"]].copy()
fold_assignments["evaluation_fold"] = 0
for fold, (_, eval_indices) in enumerate(fold_indices, 1):
    fold_assignments.loc[eval_indices, "evaluation_fold"] = fold
fold_assignments.to_csv(Path(OUTPUT_ROOT) / "fold_assignments.csv", index=False)
print("Task:", PROBLEM_TYPE, "| Classes:", NUM_LABELS)
print("Combined documents:", len(training_df), "| Folds:", N_SPLITS)
for fold, (train_indices, eval_indices) in enumerate(fold_indices, 1):
    print(f"Fold {fold}: train={len(train_indices)}, validation={len(eval_indices)}")
# 保留数据，不自动去重；重复摘要可能跨折，需在解释指标时注意。
text_keys = training_df["training_text"].str.replace(r"\s+", " ", regex=True).str.strip()
print("Repeated texts (after first):", text_keys.duplicated().sum())
print("Output:", OUTPUT_ROOT)


Training GPU: NVIDIA GeForce RTX 5080 | Device: cuda:0
Task: single_label_classification | Classes: 5
Combined documents: 2015 | Folds: 5
Fold 1: train=1612, validation=403
Fold 2: train=1612, validation=403
Fold 3: train=1612, validation=403
Fold 4: train=1612, validation=403
Fold 5: train=1612, validation=403
Repeated texts (after first): 9
Output: /home/amyliu/Desktop/GR/Rheology/results/rheology_5fold


In [7]:
def calculate_classification_metrics(logits, true_labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted_labels = np.argmax(logits, axis=1)
    true_labels = np.asarray(true_labels, dtype=np.int64)
    classes = list(range(NUM_LABELS))
    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(true_labels, predicted_labels, labels=classes, average="macro", zero_division=0),
        "micro_f1": f1_score(true_labels, predicted_labels, labels=classes, average="micro", zero_division=0),
        "macro_precision": precision_score(true_labels, predicted_labels, labels=classes, average="macro", zero_division=0),
        "macro_recall": recall_score(true_labels, predicted_labels, labels=classes, average="macro", zero_division=0),
    }


In [8]:
def evaluate_model(model_name, training_df, validation_df, fold):
    set_seed(TRAINING_SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def prepare_dataset(frame):
        dataset = Dataset.from_dict({
            "text": frame["training_text"].tolist(),
            "labels": frame["labels"].astype("int64").tolist(),
        })
        dataset = dataset.map(
            lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
            batched=True, remove_columns=["text"],
        )
        dataset.set_format("torch")
        if dataset[0]["labels"].dtype != torch.int64:
            raise TypeError("CrossEntropyLoss requires int64 class indices")
        return dataset

    train_dataset = prepare_dataset(training_df)
    eval_dataset = prepare_dataset(validation_df)
    output_dir = Path(OUTPUT_ROOT) / model_name.replace("/", "__") / f"fold_{fold}"
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS, problem_type=PROBLEM_TYPE,
        id2label=id_to_label, label2id=label_to_id,
    ).to(DEVICE)
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        optim="adamw_torch", lr_scheduler_type="linear", warmup_ratio=0.0,
        gradient_accumulation_steps=1,
        eval_strategy="no", save_strategy="no", logging_strategy="no",
        seed=TRAINING_SEED, data_seed=TRAINING_SEED,
        report_to="none", disable_tqdm=True, fp16=False, bf16=False, use_cpu=False,
    )
    trainer = Trainer(
        model=model, args=training_args, train_dataset=train_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding="longest"),
    )
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    if trainer.args.device.type != "cuda" or "5080" not in torch.cuda.get_device_name(trainer.args.device):
        raise RuntimeError(f"Trainer did not select RTX 5080: {trainer.args.device}")
    print("Model:", model_name, "| Fold:", fold, "| GPU:", torch.cuda.get_device_name(trainer.args.device), flush=True)
    try:
        torch.cuda.synchronize()
        start = time.perf_counter()
        train_result = trainer.train()
        torch.cuda.synchronize()
        training_hours = (time.perf_counter() - start) / 3600
        prediction = trainer.predict(eval_dataset)
        logits = prediction.predictions
        if isinstance(logits, tuple):
            logits = logits[0]
        metrics = calculate_classification_metrics(logits, prediction.label_ids)
        output_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(output_dir / "trained_model"))
        tokenizer.save_pretrained(str(output_dir / "trained_model"))
        np.savez_compressed(
            output_dir / "predictions.npz",
            document_ids=validation_df["document_id"].to_numpy(dtype=str),
            true_labels=prediction.label_ids, logits=logits,
            predicted_labels=np.argmax(logits, axis=1),
        )
        (output_dir / "training_args.json").write_text(training_args.to_json_string(), encoding="utf-8")
        trainer.state.save_to_json(str(output_dir / "trainer_state.json"))
        row = {
            "dataset": "rheology", "model": model_name, "fold": fold, **metrics,
            "training_loss": float(train_result.training_loss),
            "eval_loss": float(prediction.metrics["test_loss"]),
            "training_hours": training_hours,
            "train_size": len(train_dataset), "validation_size": len(eval_dataset),
            "seed": TRAINING_SEED, "epochs": NUM_EPOCHS,
            "max_length": MAX_LENGTH, "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
            "num_labels": NUM_LABELS, "problem_type": PROBLEM_TYPE,
        }
        temporary = output_dir / "metrics.tmp"
        temporary.write_text(json.dumps(row, indent=2), encoding="utf-8")
        temporary.replace(output_dir / "metrics.json")
        print(f"Saved | Accuracy={metrics['accuracy']:.4f} | Macro F1={metrics['macro_f1']:.4f}", flush=True)
        return row
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()


In [9]:
# 运行此单元格启动七个模型 × 五折；每次调用都重新加载预训练权重。
all_results = []
for model_name in MODELS:
    for fold, (train_indices, eval_indices) in enumerate(fold_indices, 1):
        row = evaluate_model(
            model_name,
            training_df.iloc[train_indices].reset_index(drop=True),
            training_df.iloc[eval_indices].reset_index(drop=True),
            fold=fold,
        )
        all_results.append(row)
        results_df = pd.DataFrame(all_results)
        temporary = Path(RESULT_CSV).with_suffix(".tmp")
        results_df.to_csv(temporary, index=False)
        temporary.replace(RESULT_CSV)

real_model_ranking = results_df.groupby("model", as_index=False).agg(
    completed_folds=("fold", "nunique"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    micro_f1_mean=("micro_f1", "mean"), micro_f1_std=("micro_f1", "std"),
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    training_hours_total=("training_hours", "sum"),
)
real_model_ranking = real_model_ranking.loc[
    real_model_ranking["completed_folds"].eq(N_SPLITS)
].copy()
real_model_ranking["real_rank"] = real_model_ranking["macro_f1_mean"].rank(
    ascending=False, method="min",
).astype(int)
real_model_ranking = real_model_ranking.sort_values(["real_rank", "model"]).reset_index(drop=True)
real_model_ranking.to_csv(Path(OUTPUT_ROOT) / "rheology_real_model_ranking.csv", index=False)
display(real_model_ranking)
print("Saved:", RESULT_CSV)


BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecate

Model: google-bert/bert-base-uncased | Fold: 1 | GPU: NVIDIA GeForce RTX 5080
Saved | Accuracy=0.7767 | Macro F1=0.7207


BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecate

Model: google-bert/bert-base-uncased | Fold: 2 | GPU: NVIDIA GeForce RTX 5080
Saved | Accuracy=0.7419 | Macro F1=0.6641


BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecate

Model: google-bert/bert-base-uncased | Fold: 3 | GPU: NVIDIA GeForce RTX 5080
Saved | Accuracy=0.7792 | Macro F1=0.6742


... (62 outputs left out)

BertForSequenceClassification LOAD REPORT from: recobo/chemical-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

Model: recobo/chemical-bert-uncased | Fold: 5 | GPU: NVIDIA GeForce RTX 5080
Saved | Accuracy=0.7618 | Macro F1=0.7273


,model,completed_folds,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,accuracy_mean,accuracy_std,training_hours_total,real_rank
0,recobo/agriculture-bert-uncased,5,0.781070,0.025743,0.803970,0.016830,0.803970,0.016830,0.018991,1
1,allenai/scibert_scivocab_uncased,5,0.779579,0.008061,0.802978,0.013200,0.802978,0.013200,0.019498,2
2,recobo/chemical-bert-uncased,5,0.729972,0.012436,0.770223,0.005146,0.770223,0.005146,0.018913,3
3,Lianglab/PharmBERT-uncased,5,0.723304,0.016389,0.775682,0.017945,0.775682,0.017945,0.020181,4
4,google-bert/bert-base-uncased,5,0.678549,0.052768,0.766749,0.019773,0.766749,0.019773,0.020255,5
5,ahmedrachid/FinancialBERT,5,0.659315,0.028792,0.716625,0.022222,0.716625,0.022222,0.019908,6
6,nlpaueb/legal-bert-base-uncased,5,0.640896,0.056043,0.742432,0.021302,0.742432,0.021302,0.019905,7


Saved: /home/amyliu/Desktop/GR/Rheology/results/rheology_5fold/rheology_5fold_results.csv
